In [ ]:
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cmocean.cm as cmo
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import dbof.dataset_creation.zarr_dataset_global as zarr_dataset
import dbof.dataset_creation.zarr_grid_global    as zarr_grid
import dbof.io.filesystems as filesystems

from dbof.global_dataset_creation.subset_definitions import get_subset_definition
from dbof.global_dataset_creation.runtime import expand_channels_with_suffixes

# Assess GLOBAL dataset — Unified Pipeline (`generate-global-gen`)

This notebook inspects any subset written by `generate-global-gen`.
Set `SUBSET` below to choose which subset to inspect.

All subsets are surface-level `(T, C, H, W)` and use `GlobalZarrDatasetReader`.
For the DEPTH pipeline, depth-derived diagnostics are computed lazily from the
full water column and reduced to 2D before writing — the Zarr stores contain
only surface fields.

### Depth suffixes (DEPTH pipeline)
Fields with depth variants use the suffixes:
- `_sfc` — z = 0 (surface, k=0)
- `_z25m` — z = 25 m (fixed depth)
- `_mld` — interpolated to the mixed-layer depth
- `_mld_mean` — thickness-weighted mean over 0 <= z <= MLD

### Surface subsets (SURF / OSN)

| Subset | Channels | Output zarr |
|---|---|---|
| `native_fields` | `Theta`, `Salt`, `Eta`, `U`, `V`, `W`, `oceTAUX`, `oceTAUY`, `SIarea` | `native_fields.zarr` |
| `frontal_structure` | `gradsalt2`, `gradtheta2`, `gradeta2`, `gradb2`, `gradrho2`, `turner_angle` | `frontal_structure.zarr` |
| `kinematic` | `relative_vorticity`, `strain_n/s`, `strain_mag`, `divergence`, `coriolis_f`, `rossby_number`, `okubo_weiss` | `kinematic.zarr` |
| `frontogenesis` | `frontogenesis_tendency`, `ug`, `vg`, `frontogenesis_geo`, `frontogenesis_ageo` | `frontogenesis.zarr` |

### Depth subsets (DEPTH)

| Subset | Channels | Output zarr |
|---|---|---|
| `stratification` | `mixed_layer_depth`, `N2_sfc/z25m/mld/mld_mean`, `ml_heat_content` | `stratification.zarr` |
| `vertical_shear` | `vertical_shear_sfc/z25m/mld/mld_mean`, `Ri_sfc/z25m/mld/mld_mean` | `vertical_shear.zarr` |
| `mixing_parameters` | `Fr/Ro/Bu_sfc/z25m/mld/mld_mean` | `mixing_parameters.zarr` |
| `ertel_pv` | `ertel_pv[_vertical/_tilt]_sfc/z25m/mld/mld_mean` (12 channels) | `ertel_pv.zarr` |
| `buoyancy_fluxes` | `uB/vB/wB_sfc/z25m/mld/mld_mean` (12 channels) | `buoyancy_fluxes.zarr` |
| `surface_wind` | `wind_stress_curl`, `ekman_pumping`, `u/v_ekman`, `oceTAUX/Y`, `oceQnet` | `surface_wind.zarr` |
| `energetics` | `KE_sfc/z25m/mld/mld_mean` | `energetics.zarr` |
| `native_fields` | `Theta/Salt/Eta/U/V/W_sfc` | `native_fields.zarr` |
| `icearea` | `SIarea` | `icearea.zarr` |
| `frontal_structure` | gradient fields x depths | `frontal_structure.zarr` |
| `kinematic` | vorticity/strain fields x depths | `kinematic.zarr` |
| `frontogenesis` | frontogenesis fields x depths | `frontogenesis.zarr` |

## Dataset access config

Parameters are loaded from `configs/data_access/global_unified.yaml`.
Update `run_id` and `date_prefix` in that file to match the session you want
to inspect.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# USER SETTINGS
# ════════════════════════════════════════════════════════════════════════════════
# Set SUBSET to any valid subset for the pipeline.
#
# Depth subsets (DEPTH pipeline):
#   'stratification', 'vertical_shear', 'mixing_parameters', 'ertel_pv',
#   'buoyancy_fluxes', 'surface_wind', 'energetics',
#   'frontal_structure', 'kinematic', 'frontogenesis',
#   'native_fields', 'icearea'
#
# Surface subsets (SURF / OSN pipeline):
#   'native_fields', 'frontal_structure', 'kinematic', 'frontogenesis'
# ════════════════════════════════════════════════════════════════════════════════
SUBSET = 'native_fields'
# ════════════════════════════════════════════════════════════════════════════════

# --- Load YAML (S3 coordinates only — no variable lists) ---
path_to_config = "../../configs/data_access/global_unified.yaml"
with open(path_to_config) as f:
    cfg = yaml.safe_load(f)

pipeline = cfg.get("pipeline", "DEPTH")

data_cfg = cfg["data_access"]
grid_cfg = cfg["grid_access"]

bucket      = data_cfg["bucket"]
folder      = data_cfg["folder"]
s3_endpoint = data_cfg["s3_endpoint"]
run_id      = data_cfg["run_id"]
date_prefix = data_cfg["date_prefix"]

# --- Get channel lists from code (single source of truth) ---
defn = get_subset_definition(pipeline, SUBSET)

dataset_name   = defn["dataset_name"]
model_channels = list(defn["model_data_feature_channels"])

# Expand compute channels with depth suffixes (same logic as generate_global_GEN.py).
compute_channels = expand_channels_with_suffixes(
    defn["compute_features_channels"],
    depth_suffixes=defn.get("depth_suffixes"),
    extra_channels=defn.get("extra_channels"),
)
all_channels = model_channels + compute_channels

grid_bucket       = grid_cfg["bucket"]
grid_folder       = grid_cfg["folder"]
grid_s3_endpoint  = grid_cfg["s3_endpoint"]
grid_dataset_name = grid_cfg["dataset_name"]

print(f"Pipeline      : {pipeline}")
print(f"Subset        : {SUBSET}")
print(f"Snapshot      : s3://{bucket}/{folder}/{run_id}/{date_prefix}/{dataset_name}")
print(f"Model channels: {model_channels}")
print(f"Compute chans : {compute_channels}")
print(f"All channels  : {all_channels}")
print(f"Grid          : s3://{grid_bucket}/{grid_folder}/{grid_dataset_name}")

# Load the data

In [ ]:
# ── Reader ──────────────────────────────────────────────────────────────────────
fs, fs_synch = filesystems.create_s3_filesystems(s3_endpoint)

reader = zarr_dataset.GlobalZarrDatasetReader(
    bucket=bucket,
    folder=folder,
    run_id=run_id,
    dataset_name=dataset_name,
    date_prefix=date_prefix,
    fs=fs,
)
print(reader)
print(f"Channels  : {reader.channel_names}")
print(f"Timesteps : {reader.n_timesteps}")
print(f"Shape     : {reader.shape}  (T, C, H, W)")

# ── Grid reader ────────────────────────────────────────────────────────────────
fs_grid, _ = filesystems.create_s3_filesystems(grid_s3_endpoint)

grid_reader = zarr_grid.GlobalGridZarrReader(
    bucket=grid_bucket,
    folder=grid_folder,
    dataset_name=grid_dataset_name,
    fs=fs_grid,
)
print(grid_reader)

XC = grid_reader.lon
YC = grid_reader.lat
print(f"Grid loaded: XC {XC.shape}, lon [{XC.min():.1f}, {XC.max():.1f}]  "
      f"YC {YC.shape}, lat [{YC.min():.1f}, {YC.max():.1f}]")

DS_GLOBAL = 20
XC_g = XC[::DS_GLOBAL, ::DS_GLOBAL]
YC_g = YC[::DS_GLOBAL, ::DS_GLOBAL]

In [ ]:
print(f"n_timesteps : {reader.n_timesteps}")
print(f"time array  : {reader.time[:]}")

# Colormap / label configuration

All channels are registered here.  Only channels present in the selected subset
will be plotted.

In [ ]:
# ── Colormap and physical label for every possible channel ────────────────────
# cmo colormaps: https://matplotlib.org/cmocean/
CMAP_CFG = {
    # --- native model variables (surface-only, no suffix) ---
    'Theta':    ('thermal', 'Potential temperature (\u00b0C)'),
    'Salt':     ('haline',  'Salinity (psu)'),
    'Eta':      ('gray',    'Sea surface height (m)'),
    'U':        ('balance', 'Zonal velocity U (m s\u207b\u00b9)'),
    'V':        ('balance', 'Meridional velocity V (m s\u207b\u00b9)'),
    'W':        ('balance', 'Vertical velocity W (m s\u207b\u00b9)'),
    'SIarea':   ('ice',     'Sea-ice area fraction (0\u20131)'),
    'oceTAUX':  ('balance', 'Zonal wind stress \u03c4\u02e3 (N m\u207b\u00b2)'),
    'oceTAUY':  ('balance', 'Meridional wind stress \u03c4\u02b8 (N m\u207b\u00b2)'),

    # --- native model variables (depth-suffixed) ---
    'Theta_sfc':      ('thermal', 'Potential temperature (\u00b0C)'),
    'Theta_z25m':     ('thermal', 'Potential temperature at 25 m (\u00b0C)'),
    'Theta_mld':      ('thermal', 'Potential temperature at MLD (\u00b0C)'),
    'Theta_mld_mean': ('thermal', 'Potential temperature ML mean (\u00b0C)'),
    'Salt_sfc':       ('haline',  'Salinity (psu)'),
    'Salt_z25m':      ('haline',  'Salinity at 25 m (psu)'),
    'Salt_mld':       ('haline',  'Salinity at MLD (psu)'),
    'Salt_mld_mean':  ('haline',  'Salinity ML mean (psu)'),
    'U_sfc':          ('balance', 'Zonal velocity U (m s\u207b\u00b9)'),
    'U_z25m':         ('balance', 'Zonal velocity U at 25 m (m s\u207b\u00b9)'),
    'U_mld':          ('balance', 'Zonal velocity U at MLD (m s\u207b\u00b9)'),
    'U_mld_mean':     ('balance', 'Zonal velocity U ML mean (m s\u207b\u00b9)'),
    'V_sfc':          ('balance', 'Meridional velocity V (m s\u207b\u00b9)'),
    'V_z25m':         ('balance', 'Meridional velocity V at 25 m (m s\u207b\u00b9)'),
    'V_mld':          ('balance', 'Meridional velocity V at MLD (m s\u207b\u00b9)'),
    'V_mld_mean':     ('balance', 'Meridional velocity V ML mean (m s\u207b\u00b9)'),
    'W_sfc':          ('balance', 'Vertical velocity W (m s\u207b\u00b9)'),
    'W_z25m':         ('balance', 'Vertical velocity W at 25 m (m s\u207b\u00b9)'),
    'W_mld':          ('balance', 'Vertical velocity W at MLD (m s\u207b\u00b9)'),
    'W_mld_mean':     ('balance', 'Vertical velocity W ML mean (m s\u207b\u00b9)'),
    'Eta_sfc':        ('gray',    'Sea surface height (m)'),

    # --- stratification ---
    'mixed_layer_depth': ('deep',    'Mixed layer depth (m)'),
    'N2_sfc':            ('tempo',   'N\u00b2 surface (s\u207b\u00b2)'),
    'N2_z25m':           ('tempo',   'N\u00b2 at 25 m (s\u207b\u00b2)'),
    'N2_mld':            ('tempo',   'N\u00b2 at MLD (s\u207b\u00b2)'),
    'N2_mld_mean':       ('tempo',   'N\u00b2 ML mean (s\u207b\u00b2)'),
    'ml_heat_content':   ('thermal', 'ML heat content (J m\u207b\u00b2)'),

    # --- vertical shear ---
    'vertical_shear_sfc':      ('speed', 'Shear |S| surface (s\u207b\u00b9)'),
    'vertical_shear_z25m':     ('speed', 'Shear |S| at 25 m (s\u207b\u00b9)'),
    'vertical_shear_mld':      ('speed', 'Shear |S| at MLD (s\u207b\u00b9)'),
    'vertical_shear_mld_mean': ('speed', 'Shear |S| ML mean (s\u207b\u00b9)'),
    'Ri_sfc':                  ('curl',  'Ri surface'),
    'Ri_z25m':                 ('curl',  'Ri at 25 m'),
    'Ri_mld':                  ('curl',  'Ri at MLD'),
    'Ri_mld_mean':             ('curl',  'Ri ML mean'),

    # --- mixing parameters ---
    'Fr_sfc':        ('amp',     'Fr surface'),
    'Fr_z25m':       ('amp',     'Fr at 25 m'),
    'Fr_mld':        ('amp',     'Fr at MLD'),
    'Fr_mld_mean':   ('amp',     'Fr ML mean'),
    'Ro_sfc':        ('balance', 'Ro surface'),
    'Ro_z25m':       ('balance', 'Ro at 25 m'),
    'Ro_mld':        ('balance', 'Ro at MLD'),
    'Ro_mld_mean':   ('balance', 'Ro ML mean'),
    'Bu_sfc':        ('matter',  'Bu surface'),
    'Bu_z25m':       ('matter',  'Bu at 25 m'),
    'Bu_mld':        ('matter',  'Bu at MLD'),
    'Bu_mld_mean':   ('matter',  'Bu ML mean'),

    # --- Ertel PV ---
    'ertel_pv_sfc':              ('curl', 'Ertel PV surface (s\u207b\u00b3)'),
    'ertel_pv_z25m':             ('curl', 'Ertel PV at 25 m (s\u207b\u00b3)'),
    'ertel_pv_mld':              ('curl', 'Ertel PV at MLD (s\u207b\u00b3)'),
    'ertel_pv_mld_mean':         ('curl', 'Ertel PV ML mean (s\u207b\u00b3)'),
    'ertel_pv_vertical_sfc':     ('curl', 'PV vertical surface (s\u207b\u00b3)'),
    'ertel_pv_vertical_z25m':    ('curl', 'PV vertical at 25 m (s\u207b\u00b3)'),
    'ertel_pv_vertical_mld':     ('curl', 'PV vertical at MLD (s\u207b\u00b3)'),
    'ertel_pv_vertical_mld_mean':('curl', 'PV vertical ML mean (s\u207b\u00b3)'),
    'ertel_pv_tilt_sfc':         ('curl', 'PV tilt surface (s\u207b\u00b3)'),
    'ertel_pv_tilt_z25m':        ('curl', 'PV tilt at 25 m (s\u207b\u00b3)'),
    'ertel_pv_tilt_mld':         ('curl', 'PV tilt at MLD (s\u207b\u00b3)'),
    'ertel_pv_tilt_mld_mean':    ('curl', 'PV tilt ML mean (s\u207b\u00b3)'),

    # --- advective buoyancy fluxes ---
    'uB_sfc':      ('balance', 'uB surface (m\u00b2 s\u207b\u00b3)'),
    'uB_z25m':     ('balance', 'uB at 25 m (m\u00b2 s\u207b\u00b3)'),
    'uB_mld':      ('balance', 'uB at MLD (m\u00b2 s\u207b\u00b3)'),
    'uB_mld_mean': ('balance', 'uB ML mean (m\u00b2 s\u207b\u00b3)'),
    'vB_sfc':      ('balance', 'vB surface (m\u00b2 s\u207b\u00b3)'),
    'vB_z25m':     ('balance', 'vB at 25 m (m\u00b2 s\u207b\u00b3)'),
    'vB_mld':      ('balance', 'vB at MLD (m\u00b2 s\u207b\u00b3)'),
    'vB_mld_mean': ('balance', 'vB ML mean (m\u00b2 s\u207b\u00b3)'),
    'wB_sfc':      ('balance', 'wB surface (m\u00b2 s\u207b\u00b3)'),
    'wB_z25m':     ('balance', 'wB at 25 m (m\u00b2 s\u207b\u00b3)'),
    'wB_mld':      ('balance', 'wB at MLD (m\u00b2 s\u207b\u00b3)'),
    'wB_mld_mean': ('balance', 'wB ML mean (m\u00b2 s\u207b\u00b3)'),

    # --- surface wind diagnostics ---
    'oceQnet':          ('thermal', 'Net heat flux (W m\u207b\u00b2)'),
    'wind_stress_curl': ('balance', 'Wind stress curl (N m\u207b\u00b3)'),
    'ekman_pumping':    ('balance', 'Ekman pumping (m s\u207b\u00b9)'),
    'u_ekman':          ('balance', 'Ekman transport u (m s\u207b\u00b9)'),
    'v_ekman':          ('balance', 'Ekman transport v (m s\u207b\u00b9)'),

    # --- energetics ---
    'KE_sfc':      ('speed', 'Kinetic energy surface (m\u00b2 s\u207b\u00b2)'),
    'KE_z25m':     ('speed', 'Kinetic energy at 25 m (m\u00b2 s\u207b\u00b2)'),
    'KE_mld':      ('speed', 'Kinetic energy at MLD (m\u00b2 s\u207b\u00b2)'),
    'KE_mld_mean': ('speed', 'Kinetic energy ML mean (m\u00b2 s\u207b\u00b2)'),

    # --- frontal structure (surface-only, no suffix) ---
    'gradb2':       ('amp',     'Buoyancy gradient magnitude (m s\u207b\u00b2)'),
    'gradsalt2':    ('amp',     'Salinity gradient magnitude\u00b2 (psu\u00b2 m\u207b\u00b2)'),
    'gradtheta2':   ('amp',     'Temperature gradient magnitude (\u00b0C m\u207b\u00b9)'),
    'gradeta2':     ('amp',     'SSH gradient magnitude (m m\u207b\u00b9)'),
    'gradrho2':     ('amp',     'Density gradient magnitude\u00b2 (kg\u00b2 m\u207b\u2076)'),
    'turner_angle': ('balance', 'Turner angle (\u00b0)'),

    # --- frontal structure (depth-suffixed) ---
    'gradb2_sfc':           ('amp',     'Buoyancy grad mag surface (m s\u207b\u00b2)'),
    'gradb2_z25m':          ('amp',     'Buoyancy grad mag 25 m (m s\u207b\u00b2)'),
    'gradb2_mld':           ('amp',     'Buoyancy grad mag MLD (m s\u207b\u00b2)'),
    'gradb2_mld_mean':      ('amp',     'Buoyancy grad mag ML mean (m s\u207b\u00b2)'),
    'gradtheta2_sfc':       ('amp',     'Temp grad mag surface (\u00b0C m\u207b\u00b9)'),
    'gradtheta2_z25m':      ('amp',     'Temp grad mag 25 m (\u00b0C m\u207b\u00b9)'),
    'gradtheta2_mld':       ('amp',     'Temp grad mag MLD (\u00b0C m\u207b\u00b9)'),
    'gradtheta2_mld_mean':  ('amp',     'Temp grad mag ML mean (\u00b0C m\u207b\u00b9)'),
    'gradsalt2_sfc':        ('amp',     'Salt grad mag\u00b2 surface (psu\u00b2 m\u207b\u00b2)'),
    'gradsalt2_z25m':       ('amp',     'Salt grad mag\u00b2 25 m (psu\u00b2 m\u207b\u00b2)'),
    'gradsalt2_mld':        ('amp',     'Salt grad mag\u00b2 MLD (psu\u00b2 m\u207b\u00b2)'),
    'gradsalt2_mld_mean':   ('amp',     'Salt grad mag\u00b2 ML mean (psu\u00b2 m\u207b\u00b2)'),
    'gradrho2_sfc':         ('amp',     'Density grad mag\u00b2 surface (kg\u00b2 m\u207b\u2076)'),
    'gradrho2_z25m':        ('amp',     'Density grad mag\u00b2 25 m (kg\u00b2 m\u207b\u2076)'),
    'gradrho2_mld':         ('amp',     'Density grad mag\u00b2 MLD (kg\u00b2 m\u207b\u2076)'),
    'gradrho2_mld_mean':    ('amp',     'Density grad mag\u00b2 ML mean (kg\u00b2 m\u207b\u2076)'),
    'gradeta2_sfc':         ('amp',     'SSH grad mag surface (m m\u207b\u00b9)'),
    'gradeta2_z25m':        ('amp',     'SSH grad mag 25 m (m m\u207b\u00b9)'),
    'gradeta2_mld':         ('amp',     'SSH grad mag MLD (m m\u207b\u00b9)'),
    'gradeta2_mld_mean':    ('amp',     'SSH grad mag ML mean (m m\u207b\u00b9)'),
    'turner_angle_sfc':     ('balance', 'Turner angle surface (\u00b0)'),
    'turner_angle_z25m':    ('balance', 'Turner angle 25 m (\u00b0)'),
    'turner_angle_mld':     ('balance', 'Turner angle MLD (\u00b0)'),
    'turner_angle_mld_mean':('balance', 'Turner angle ML mean (\u00b0)'),

    # --- kinematic (surface-only, no suffix) ---
    'relative_vorticity': ('curl',    'Relative vorticity (s\u207b\u00b9)'),
    'strain_n':           ('curl',    'Normal strain (s\u207b\u00b9)'),
    'strain_s':           ('curl',    'Shear strain (s\u207b\u00b9)'),
    'strain_mag':         ('curl',    'Strain magnitude (s\u207b\u00b9)'),
    'divergence':         ('curl',    'Divergence (s\u207b\u00b9)'),
    'coriolis_f':         ('balance', 'Coriolis f (s\u207b\u00b9)'),
    'rossby_number':      ('balance', 'Rossby number'),
    'okubo_weiss':        ('curl',    'Okubo-Weiss parameter (s\u207b\u00b2)'),

    # --- kinematic (depth-suffixed) ---
    'relative_vorticity_sfc':      ('curl', 'Rel. vorticity surface (s\u207b\u00b9)'),
    'relative_vorticity_z25m':     ('curl', 'Rel. vorticity 25 m (s\u207b\u00b9)'),
    'relative_vorticity_mld':      ('curl', 'Rel. vorticity MLD (s\u207b\u00b9)'),
    'relative_vorticity_mld_mean': ('curl', 'Rel. vorticity ML mean (s\u207b\u00b9)'),
    'divergence_sfc':              ('curl', 'Divergence surface (s\u207b\u00b9)'),
    'divergence_z25m':             ('curl', 'Divergence 25 m (s\u207b\u00b9)'),
    'divergence_mld':              ('curl', 'Divergence MLD (s\u207b\u00b9)'),
    'divergence_mld_mean':         ('curl', 'Divergence ML mean (s\u207b\u00b9)'),
    'strain_n_sfc':                ('curl', 'Normal strain surface (s\u207b\u00b9)'),
    'strain_n_z25m':               ('curl', 'Normal strain 25 m (s\u207b\u00b9)'),
    'strain_n_mld':                ('curl', 'Normal strain MLD (s\u207b\u00b9)'),
    'strain_n_mld_mean':           ('curl', 'Normal strain ML mean (s\u207b\u00b9)'),
    'strain_s_sfc':                ('curl', 'Shear strain surface (s\u207b\u00b9)'),
    'strain_s_z25m':               ('curl', 'Shear strain 25 m (s\u207b\u00b9)'),
    'strain_s_mld':                ('curl', 'Shear strain MLD (s\u207b\u00b9)'),
    'strain_s_mld_mean':           ('curl', 'Shear strain ML mean (s\u207b\u00b9)'),
    'strain_mag_sfc':              ('curl', 'Strain mag surface (s\u207b\u00b9)'),
    'strain_mag_z25m':             ('curl', 'Strain mag 25 m (s\u207b\u00b9)'),
    'strain_mag_mld':              ('curl', 'Strain mag MLD (s\u207b\u00b9)'),
    'strain_mag_mld_mean':         ('curl', 'Strain mag ML mean (s\u207b\u00b9)'),
    'okubo_weiss_sfc':             ('curl', 'Okubo-Weiss surface (s\u207b\u00b2)'),
    'okubo_weiss_z25m':            ('curl', 'Okubo-Weiss 25 m (s\u207b\u00b2)'),
    'okubo_weiss_mld':             ('curl', 'Okubo-Weiss MLD (s\u207b\u00b2)'),
    'okubo_weiss_mld_mean':        ('curl', 'Okubo-Weiss ML mean (s\u207b\u00b2)'),

    # --- frontogenesis (surface-only, no suffix) ---
    'frontogenesis_tendency': ('curl',    'Frontogenesis tendency (s\u207b\u00b2)'),
    'ug':                     ('balance', 'Geostrophic U (m s\u207b\u00b9)'),
    'vg':                     ('balance', 'Geostrophic V (m s\u207b\u00b9)'),
    'frontogenesis_geo':      ('curl',    'Geostrophic frontogenesis (s\u207b\u00b2)'),
    'frontogenesis_ageo':     ('curl',    'Ageostrophic frontogenesis (s\u207b\u00b2)'),

    # --- frontogenesis (depth-suffixed) ---
    'frontogenesis_tendency_sfc':      ('curl', 'Frontogenesis surface (s\u207b\u00b2)'),
    'frontogenesis_tendency_z25m':     ('curl', 'Frontogenesis 25 m (s\u207b\u00b2)'),
    'frontogenesis_tendency_mld':      ('curl', 'Frontogenesis MLD (s\u207b\u00b2)'),
    'frontogenesis_tendency_mld_mean': ('curl', 'Frontogenesis ML mean (s\u207b\u00b2)'),
    'frontogenesis_geo_sfc':           ('curl', 'Geo frontogenesis surface (s\u207b\u00b2)'),
    'frontogenesis_geo_z25m':          ('curl', 'Geo frontogenesis 25 m (s\u207b\u00b2)'),
    'frontogenesis_geo_mld':           ('curl', 'Geo frontogenesis MLD (s\u207b\u00b2)'),
    'frontogenesis_geo_mld_mean':      ('curl', 'Geo frontogenesis ML mean (s\u207b\u00b2)'),
    'frontogenesis_ageo_sfc':          ('curl', 'Ageo frontogenesis surface (s\u207b\u00b2)'),
    'frontogenesis_ageo_z25m':         ('curl', 'Ageo frontogenesis 25 m (s\u207b\u00b2)'),
    'frontogenesis_ageo_mld':          ('curl', 'Ageo frontogenesis MLD (s\u207b\u00b2)'),
    'frontogenesis_ageo_mld_mean':     ('curl', 'Ageo frontogenesis ML mean (s\u207b\u00b2)'),
    'ug_sfc':      ('balance', 'Geostrophic U surface (m s\u207b\u00b9)'),
    'ug_z25m':     ('balance', 'Geostrophic U 25 m (m s\u207b\u00b9)'),
    'ug_mld':      ('balance', 'Geostrophic U MLD (m s\u207b\u00b9)'),
    'ug_mld_mean': ('balance', 'Geostrophic U ML mean (m s\u207b\u00b9)'),
    'vg_sfc':      ('balance', 'Geostrophic V surface (m s\u207b\u00b9)'),
    'vg_z25m':     ('balance', 'Geostrophic V 25 m (m s\u207b\u00b9)'),
    'vg_mld':      ('balance', 'Geostrophic V MLD (m s\u207b\u00b9)'),
    'vg_mld_mean': ('balance', 'Geostrophic V ML mean (m s\u207b\u00b9)'),
}

channels_to_plot = [ch for ch in reader.channel_names if ch in CMAP_CFG]
print(f"Plotting {len(channels_to_plot)} / {len(reader.channel_names)} channels")
if len(channels_to_plot) < len(reader.channel_names):
    missing = [ch for ch in reader.channel_names if ch not in CMAP_CFG]
    print(f"  (missing from CMAP_CFG: {missing})")

# Channels that should use a logarithmic colour scale.
LOG_SCALE_CHANNELS = {k for k in CMAP_CFG if k.startswith('N2_')}

# Diverging colormaps that should be centred at zero.
DIVERGING_CMAPS = {'balance', 'curl'}

## Notes

All output subsets are 2D surface-level fields `(T, C, H, W)`.
Depth-derived diagnostics (MLD, N\u00b2, shear, PV, etc.) were computed from the
full water column and reduced to 2D before writing.  There is no K axis to
select.

In [ ]:
print(f"Subset '{SUBSET}' \u2014 all channels are 2D surface fields.")
print(f"Available channels: {reader.channel_names}")

## Single-channel global map

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# USER SETTINGS
target   = channels_to_plot[0]
timestep = 0
# ════════════════════════════════════════════════════════════════════════════════

arr_ds = reader.get_channel_snapshot(timestep, target)[::DS_GLOBAL, ::DS_GLOBAL]

cmap_name, label = CMAP_CFG[target]
cmap = getattr(cmo, cmap_name)
finite = arr_ds[np.isfinite(arr_ds)]
vmin, vmax = np.nanpercentile(finite, [1, 99])
if target in LOG_SCALE_CHANNELS:
    pos = finite[finite > 0]
    vmin, vmax = np.nanpercentile(pos, [1, 99]) if pos.size > 0 else (1e-8, 1e-3)
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
elif cmap_name in DIVERGING_CMAPS:
    vlim = max(abs(vmin), abs(vmax))
    norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vlim, vmax=vlim)
else:
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

fig = plt.figure(figsize=(16, 8))
ax  = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())

im = ax.pcolormesh(XC_g, YC_g, arr_ds,
                   cmap=cmap, norm=norm,
                   transform=ccrs.PlateCarree(),
                   shading='nearest')
ax.add_feature(cfeature.COASTLINE, linewidth=0.6, color='k')
ax.gridlines(linewidth=0.4, color='gray', alpha=0.6)

plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04, shrink=0.7,
             label=label, orientation='horizontal')
ax.set_title(
    f'LLC4320 {target}  |  {DS_GLOBAL}\u00d7 downsampled',
    fontsize=13,
)
plt.suptitle(
    f'Global LLC4320 [{SUBSET}] \u2014 iteration {reader.time[timestep]}',
    fontsize=14, y=1.01,
)
plt.tight_layout()
plt.show()

## All-channels global overview

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# USER SETTINGS
timestep = 0
ncols    = 3
# ════════════════════════════════════════════════════════════════════════════════

nrows = int(np.ceil(len(channels_to_plot) / ncols))
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(ncols * 7, nrows * 4),
    subplot_kw={'projection': ccrs.Robinson()},
)
axes = np.array(axes).flatten()

for ax, ch in zip(axes, channels_to_plot):
    arr = reader.get_channel_snapshot(timestep, ch)[::DS_GLOBAL, ::DS_GLOBAL]
    cmap_name, label = CMAP_CFG[ch]
    cmap = getattr(cmo, cmap_name)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        ax.set_visible(False)
        continue
    vmin, vmax = np.nanpercentile(finite, [2, 98])
    if ch in LOG_SCALE_CHANNELS:
        pos = finite[finite > 0]
        vmin, vmax = np.nanpercentile(pos, [2, 98]) if pos.size > 0 else (1e-8, 1e-3)
        norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    elif cmap_name in DIVERGING_CMAPS:
        vlim = max(abs(vmin), abs(vmax))
        norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vlim, vmax=vlim)
    else:
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    im = ax.pcolormesh(XC_g, YC_g, arr,
                       cmap=cmap, norm=norm,
                       transform=ccrs.PlateCarree(),
                       shading='nearest')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, color='k')
    ax.set_title(label, fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, shrink=0.85,
                 orientation='horizontal')

for ax in axes[len(channels_to_plot):]:
    ax.set_visible(False)

plt.suptitle(
    f'Global LLC4320 [{SUBSET}]  |  '
    f'iteration {reader.time[timestep]}  |  {DS_GLOBAL}\u00d7 downsampled',
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()

## Compare depth variants of the same field

For fields with `_sfc`, `_z25m`, `_mld`, `_mld_mean` suffixes, plot all four
side by side to see how the field varies across depth definitions.

*Skip this cell if the current subset has no depth suffixes.*

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# USER SETTINGS
# Set BASE_FIELD to any field that has _sfc/_z25m/_mld/_mld_mean variants.
# Examples: 'N2', 'vertical_shear', 'Ri', 'Fr', 'Ro', 'ertel_pv',
#           'ertel_pv_vertical', 'ertel_pv_tilt', 'uB', 'vB', 'wB', 'KE'
BASE_FIELD = 'N2'
timestep   = 0
# ════════════════════════════════════════════════════════════════════════════════

depth_suffixes_plot = ['sfc', 'z25m', 'mld', 'mld_mean']
variant_channels = [f"{BASE_FIELD}_{s}" for s in depth_suffixes_plot]
available = [ch for ch in variant_channels if ch in reader.channel_names]

if not available:
    print(f"No depth variants found for '{BASE_FIELD}' in this subset.")
    print(f"Available channels: {reader.channel_names}")
else:
    ncols_v = len(available)
    fig, axes = plt.subplots(
        1, ncols_v,
        figsize=(ncols_v * 6, 5),
        subplot_kw={'projection': ccrs.Robinson()},
    )
    if ncols_v == 1:
        axes = [axes]

    for ax, ch in zip(axes, available):
        arr = reader.get_channel_snapshot(timestep, ch)[::DS_GLOBAL, ::DS_GLOBAL]
        cmap_name, label = CMAP_CFG.get(ch, ('viridis', ch))
        cmap = getattr(cmo, cmap_name, plt.cm.viridis)
        finite = arr[np.isfinite(arr)]
        if finite.size == 0:
            ax.set_visible(False)
            continue
        vmin, vmax = np.nanpercentile(finite, [2, 98])
        if ch in LOG_SCALE_CHANNELS:
            pos = finite[finite > 0]
            vmin, vmax = np.nanpercentile(pos, [2, 98]) if pos.size > 0 else (1e-8, 1e-3)
            norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
        elif cmap_name in DIVERGING_CMAPS:
            vlim = max(abs(vmin), abs(vmax))
            norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vlim, vmax=vlim)
        else:
            norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        im = ax.pcolormesh(XC_g, YC_g, arr,
                           cmap=cmap, norm=norm,
                           transform=ccrs.PlateCarree(),
                           shading='nearest')
        ax.add_feature(cfeature.COASTLINE, linewidth=0.4, color='k')
        ax.set_title(label, fontsize=10)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, shrink=0.85,
                     orientation='horizontal')

    plt.suptitle(
        f'{BASE_FIELD} depth variants  |  iteration {reader.time[timestep]}  |  {DS_GLOBAL}\u00d7 downsampled',
        fontsize=13, y=1.01,
    )
    plt.tight_layout()
    plt.show()

## Per-channel statistics

Quick sanity check: min/max/mean/NaN fraction for each channel.

In [ ]:
timestep = 0
print(f"{'Channel':<30s}  {'Min':>12s}  {'Max':>12s}  {'Mean':>12s}  {'NaN %':>7s}")
print("\u2500" * 80)
for ch in reader.channel_names:
    arr = reader.get_channel_snapshot(timestep, ch)
    finite = arr[np.isfinite(arr)]
    nan_pct = 100.0 * (1.0 - finite.size / arr.size)
    if finite.size > 0:
        print(f"{ch:<30s}  {finite.min():>12.4g}  {finite.max():>12.4g}  {finite.mean():>12.4g}  {nan_pct:>6.1f}%")
    else:
        print(f"{ch:<30s}  {'all NaN':>12s}  {'':>12s}  {'':>12s}  {nan_pct:>6.1f}%")

## Regional plots

### Region selection by lat/lon

The grid (XC, YC) is loaded \u2014 you can select any region by geographic
coordinates.  The nearest pixel to your requested **(lat, lon)** centre is
found automatically, and the bounding box is derived from an approximate
km-to-pixel conversion (LLC4320 native resolution \u2248 1/48\u00b0 \u2248 2.3 km at
mid-latitudes).

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# USER SETTINGS \u2014 edit these to choose your region
# ════════════════════════════════════════════════════════════════════════════════
LAT_CENTER = 36.0       # degrees north
LON_CENTER = -72.0      # degrees east  (use negative for west)
HALF_H_KM  = 1000.0     # approximate half-height of region (km)
HALF_W_KM  = 2000.0     # approximate half-width  of region (km)
# ════════════════════════════════════════════════════════════════════════════════

H, W = reader.rectangular_shape

lon_q = LON_CENTER % 360 if float(XC.max()) > 180 else LON_CENTER

dist = np.sqrt((XC - lon_q) ** 2 + (YC - LAT_CENTER) ** 2)
cy, cx = np.unravel_index(int(np.nanargmin(dist)), dist.shape)
print(f"Nearest pixel to ({LAT_CENTER}\u00b0N, {LON_CENTER}\u00b0E): y={cy}, x={cx}  "
      f"[actual: lat={float(YC[cy,cx]):.3f}\u00b0N, lon={float(XC[cy,cx]):.3f}\u00b0E]")

KM_PER_PIX = 2.3
HALF_H_PIX = int(HALF_H_KM / KM_PER_PIX)
HALF_W_PIX = int(HALF_W_KM / KM_PER_PIX)

y0 = max(0, cy - HALF_H_PIX)
y1 = min(H, cy + HALF_H_PIX)
x0 = max(0, cx - HALF_W_PIX)
x1 = min(W, cx + HALF_W_PIX)

XC_reg = XC[y0:y1, x0:x1]
YC_reg = YC[y0:y1, x0:x1]

print(f"Region: rows [{y0}:{y1}], cols [{x0}:{x1}]  ->  {y1-y0} x {x1-x0} pixels")
print(f"Geographic extent: lon [{float(XC_reg.min()):.2f}, {float(XC_reg.max()):.2f}]  "
      f"lat [{float(YC_reg.min()):.2f}, {float(YC_reg.max()):.2f}]")

In [ ]:
# \u2500\u2500 Global context map with selected region outlined \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
context_ch = channels_to_plot[0]
ctx_ds = reader.get_channel_snapshot(0, context_ch)[::DS_GLOBAL, ::DS_GLOBAL]

ctx_cmap_name, ctx_label = CMAP_CFG[context_ch]
vmin_c, vmax_c = np.nanpercentile(ctx_ds[np.isfinite(ctx_ds)], [2, 98])

fig = plt.figure(figsize=(18, 9))
ax  = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())

im = ax.pcolormesh(XC_g, YC_g, ctx_ds,
                   cmap=getattr(cmo, ctx_cmap_name), vmin=vmin_c, vmax=vmax_c,
                   transform=ccrs.PlateCarree(), shading='nearest')
plt.colorbar(im, ax=ax, fraction=0.025, pad=0.04, shrink=0.7,
             label=ctx_label, orientation='horizontal')
ax.add_feature(cfeature.COASTLINE, linewidth=0.6, color='k')
ax.gridlines(linewidth=0.4, color='gray', alpha=0.5)

pc = ccrs.PlateCarree()
ax.plot(XC[y0, x0:x1],   YC[y0, x0:x1],   'r-', lw=2, transform=pc)
ax.plot(XC[y1-1, x0:x1], YC[y1-1, x0:x1], 'r-', lw=2, transform=pc)
ax.plot(XC[y0:y1, x0],   YC[y0:y1, x0],   'r-', lw=2, transform=pc)
ax.plot(XC[y0:y1, x1-1], YC[y0:y1, x1-1], 'r-', lw=2, transform=pc)
ax.plot(float(XC[cy, cx]), float(YC[cy, cx]),
        'r+', markersize=14, markeredgewidth=2.5, transform=pc, zorder=6)

ax.set_title(
    f'Global [{SUBSET}] \u2014 {context_ch} ({DS_GLOBAL}\u00d7 downsampled) \u2014 selected region (red box)\n'
    f'centre: ({LAT_CENTER}\u00b0N, {LON_CENTER}\u00b0E)',
    fontsize=12,
)
plt.tight_layout()
plt.show()

In [ ]:
# \u2500\u2500 Regional subplots \u2014 all channels \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
# ════════════════════════════════════════════════════════════════════════════════
# USER SETTINGS
timestep = 0
ncols    = 3
# ════════════════════════════════════════════════════════════════════════════════

nrows = int(np.ceil(len(channels_to_plot) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 7, nrows * 6))
axes = np.array(axes).flatten()

for ax, ch in zip(axes, channels_to_plot):
    arr = reader.get_channel_snapshot(timestep, ch)[y0:y1, x0:x1]
    cmap_name, label = CMAP_CFG[ch]
    cmap = getattr(cmo, cmap_name)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        ax.set_visible(False)
        continue
    vmin, vmax = np.nanpercentile(finite, [1, 99])
    if ch in LOG_SCALE_CHANNELS:
        pos = finite[finite > 0]
        vmin, vmax = np.nanpercentile(pos, [1, 99]) if pos.size > 0 else (1e-8, 1e-3)
        norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    elif cmap_name in DIVERGING_CMAPS:
        vlim = max(abs(vmin), abs(vmax))
        norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vlim, vmax=vlim)
    else:
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    im = ax.pcolormesh(XC_reg, YC_reg, arr,
                       cmap=cmap, norm=norm,
                       shading='nearest')
    ax.set_title(label, fontsize=12)
    ax.set_xlabel('Longitude (\u00b0E)', fontsize=9)
    ax.set_ylabel('Latitude (\u00b0N)',  fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, shrink=0.9)

for ax in axes[len(channels_to_plot):]:
    ax.set_visible(False)

plt.suptitle(
    f'Regional [{SUBSET}]  |  '
    f'centre ({LAT_CENTER}\u00b0N, {LON_CENTER}\u00b0E)  |  '
    f'lon [{float(XC_reg.min()):.1f}\u00b0, {float(XC_reg.max()):.1f}\u00b0]  '
    f'lat [{float(YC_reg.min()):.1f}\u00b0, {float(YC_reg.max()):.1f}\u00b0]',
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()